# Cross-track position vs. time diagnostic

**Why this exists.** The pooled quasi-BRDF drift model fits one shared drift rate γ while each PICS keeps its own brightness/view-angle terms. That framework assumes the *acquisition geometry* is not itself trending with time — otherwise a slow geometric wander could masquerade as (or cancel) a radiometric drift. Appendix B already checks that cos(SZA) is not correlated with time; this notebook checks the **view side** of the geometry: the **cross-track position** of each patch within the EMIT swath.

**What "cross-track position" is.** EMIT L1B/L2A products are delivered in native pushbroom sensor geometry — arrays of shape `(downtrack lines, crosstrack samples, bands)` where the crosstrack axis runs across the 1242-element detector, nadir near the centre. So the **crosstrack sample column index** of a pixel *is* its cross-track detector position. The extractor records it per in-patch pixel as `col` (see `single_patch_cube_extract.py`, `cube["col"] = c0 + wc`), and `write_cube_netcdf` persists it into every cached cube `.nc` alongside `row`, `lon`, `lat`, `sza`, `vza`.

**Why a dedicated notebook (not an appendix / repo script).** The per-scene mean cross-track position is *not* summarised in the retained per-scene tables — `geometry_stats_from_cube` rolls up SZA and VZA only. Recovering `col` therefore requires reading the raw pixel cubes in `data/pixel_cubes/`, which on some machines have been pruned after caching. Run this **where the cubes still live** (e.g. the work machine).

**Cross-track vs. the VZA proxy.** VZA (to-sensor zenith) *is* retained and is roughly a monotonic function of |cross-track offset from nadir| — but it is unsigned, so it cannot tell which side of nadir a scene fell on. Mean `col` is signed and is the faithful quantity; this notebook also cross-checks `col` against VZA to confirm the proxy behaves as expected.

> Runs off `data/brdf/brdf_design.csv` (the QC-surviving, fitted scenes with the shared clock `t`) and the cubes at `paths.pixel_cube(site, patch, timestamp)`. Complete the BRDF assemble/drift stage first so the design table exists.

In [ ]:
# Repo-relative bootstrap: find the repo root (holds run_config.yaml) and import.
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / "run_config.yaml").exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from emit_pics import paths
from emit_pics.run_config import load_config
from emit_pics.viz import style

FILL = -9999.0                # EMIT geometry fill (id_matched_pairs.FILL)
NADIR_COL = 620.5             # EMIT swath centre (1242 cross-track elements); only
                              # shifts the signed offset — the correlation is invariant to it.

cfg = load_config()
paths.set_variant(cfg.output_variant)   # match the active radiometry variant
paths.set_brdf_label(cfg.brdf_label)    # read the BRDF design from its labelled namespace, if any
print("repo root:", _p)

## 1. The QC-surviving, fitted scenes

We key off `brdf_design.csv` rather than re-deriving the roster: it already lists exactly the scenes that entered the pooled drift fit (post-QC) and carries the shared clock `t` (years since the earliest scene across all sites). That guarantees the cross-track test is evaluated on the *same* scenes as the drift result.

In [ ]:
design_path = paths.brdf_design()
if not design_path.exists():
    raise SystemExit(
        f"BRDF design not found: {design_path}\n"
        "Run scripts/run_brdf_assemble.py (or the BRDF drift stage) first.")

design = pd.read_csv(design_path)
need = {"site", "patch", "timestamp", "t"}
missing = need - set(design.columns)
if missing:
    raise SystemExit(f"brdf_design.csv is missing columns: {missing}")

sites = list(design.groupby("site", sort=False).groups.keys())
print(f"{len(design)} QC-surviving scenes across {len(sites)} PICS:")
for s in sites:
    d = design[design["site"] == s]
    print(f"  {s:<14} patch {d['patch'].iloc[0]:<8} n={len(d)}")

## 2. Read the cubes → per-scene mean cross-track position

For each scene we open its cached pixel cube and average the in-patch pixels' `col` (cross-track sample index). We also compute the signed offset from swath centre (`col_offset = mean_col - NADIR_COL`; negative/positive = one side of nadir vs. the other) and the mean VZA for the cross-check in §4. Missing cubes are reported and skipped, not fatal — so a partially-pruned cache still yields a partial result you can act on.

In [ ]:
def scene_crosstrack(site, patch, ts):
    """Mean cross-track column (and mean VZA) over a scene's in-patch pixels.

    Returns a dict, or None if the cube is absent / has no pixels.
    """
    cube_path = paths.pixel_cube(site, patch, ts)
    if not cube_path.exists():
        return None
    with xr.open_dataset(cube_path) as ds:
        if "col" not in ds or ds.sizes.get("pixel", 0) == 0:
            return None
        col = np.asarray(ds["col"].values, dtype=float)
        out = {
            "n_pixels": int(col.size),
            "mean_col": float(np.mean(col)),
            "col_min": float(np.min(col)),
            "col_max": float(np.max(col)),
        }
        if "vza" in ds:
            vza = np.asarray(ds["vza"].values, dtype=float)
            vza = np.where(np.isclose(vza, FILL), np.nan, vza)
            out["mean_vza"] = float(np.nanmean(vza))
    return out


rows, missing_scenes = [], []
for _, r in design.iterrows():
    stats = scene_crosstrack(r["site"], r["patch"], r["timestamp"])
    if stats is None:
        missing_scenes.append((r["site"], r["patch"], r["timestamp"]))
        continue
    rows.append({"site": r["site"], "patch": r["patch"], "timestamp": r["timestamp"],
                 "t": r["t"], **stats})

ct = pd.DataFrame(rows)
if not ct.empty:
    ct["col_offset"] = ct["mean_col"] - NADIR_COL   # signed distance from swath centre

print(f"read {len(ct)} / {len(design)} cubes", end="")
if missing_scenes:
    print(f"  |  {len(missing_scenes)} missing (cube not on disk):")
    for s, p, t in missing_scenes[:20]:
        print(f"    - {s} {p} {t}")
    if len(missing_scenes) > 20:
        print(f"    ... and {len(missing_scenes) - 20} more")
else:
    print("  |  all cubes present.")
ct.head()

## 3. Per-PICS Spearman: mean cross-track position vs. time

A non-zero, significant ρ means the patch's cross-track position drifted monotonically across the record — a geometric confound the drift model does not absorb. We use Spearman (rank) for robustness to the small per-site n and to any non-linearity. Significance is read at the same α as the rest of the paper (`cfg.alpha`).

In [ ]:
ALPHA = cfg.alpha
results = []
for s in sites:
    d = ct[ct["site"] == s].dropna(subset=["mean_col", "t"])
    if len(d) < 3:
        results.append({"site": s, "n": len(d), "rho": np.nan, "p": np.nan})
        continue
    rho, p = spearmanr(d["t"], d["mean_col"])
    results.append({"site": s, "n": len(d), "rho": rho, "p": p})

res = pd.DataFrame(results)
res["significant"] = res["p"] < ALPHA
print(f"Spearman  mean cross-track column vs. time   (alpha = {ALPHA})")
print(res.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
n_sig = int(res["significant"].fillna(False).sum())
print(f"\n{n_sig} / {len(res)} PICS show a significant cross-track trend.")

## 4. Figure: cross-track position vs. time, per PICS

Five panels (one per PICS), Spearman ρ and p printed bottom-right, colour/marker per the manuscript site palette. This mirrors the cos(SZA)-vs-time appendix figure so the two geometry checks read side by side.

In [ ]:
smap = style.site_styles(sites)   # distinct colour+marker per site (colour-blind-safe)
rmap = {r["site"]: r for r in results}

with style.theme("manuscript"):
    fig, axes = plt.subplots(2, 3, figsize=style.figsize("double", aspect=0.62))
    axes = axes.ravel()
    for i, ax in enumerate(axes):
        if i >= len(sites):
            ax.set_visible(False)
            continue
        s = sites[i]
        d = ct[ct["site"] == s].dropna(subset=["mean_col", "t"])
        st = smap[s]
        ax.scatter(d["t"], d["mean_col"], c=[st["color"]], marker=st["marker"],
                   s=34, edgecolor="k", linewidth=0.4, zorder=3)
        ax.axhline(NADIR_COL, color="0.6", lw=0.8, ls="--", zorder=1)   # swath centre
        ax.set_title(s.replace("_", " "), fontsize=9)
        rr = rmap[s]
        ax.text(0.96, 0.04, f"$\\rho$ = {rr['rho']:+.2f}\n$p$ = {rr['p']:.2f}  (n={rr['n']})",
                transform=ax.transAxes, ha="right", va="bottom", fontsize=8)
        if i % 3 == 0:
            ax.set_ylabel("Mean cross-track column")
        if i + 3 >= len(sites):
            ax.set_xlabel("Years since first scene")
    fig.suptitle("Cross-track position (mean crosstrack sample) vs. time, per PICS",
                 fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

# To save a copy, uncomment:
# fig.savefig(pathlib.Path.home() / "Downloads" / "crosstrack_time_corr.png",
#             dpi=200, bbox_inches="tight")

## 5. Cross-check: does VZA track |cross-track offset|?

Sanity check on the earlier VZA proxy. Because nadir sits near the swath centre, mean VZA should rise roughly linearly with |col − nadir|. If this holds, the retained VZA is a usable (unsigned) stand-in when cubes are unavailable; where it *breaks* is exactly where the signed `col` adds information the proxy could not.

In [ ]:
if "mean_vza" in ct.columns and ct["mean_vza"].notna().any():
    d = ct.dropna(subset=["mean_vza", "col_offset"])
    rho, p = spearmanr(d["col_offset"].abs(), d["mean_vza"])
    with style.theme("manuscript"):
        fig, ax = plt.subplots(figsize=style.figsize("single", aspect=0.8))
        for s in sites:
            ds = d[d["site"] == s]
            st = smap[s]
            ax.scatter(ds["col_offset"].abs(), ds["mean_vza"], c=[st["color"]],
                       marker=st["marker"], s=30, edgecolor="k", linewidth=0.3,
                       label=s.replace("_", " "))
        ax.set_xlabel("|mean_col − nadir|  (cross-track samples)")
        ax.set_ylabel("Mean VZA (deg)")
        ax.set_title(f"VZA vs |cross-track offset|   (Spearman $\\rho$={rho:+.2f}, p={p:.2g})")
        ax.legend(fontsize=7, loc="best")
        fig.tight_layout()
        plt.show()
else:
    print("No VZA in the cubes read — skipping the proxy cross-check.")

## How to read this

- **All ρ non-significant (p ≥ α):** cross-track position is *not* trending with time at any PICS — the pooled drift is not confounded by a geometric wander across the swath. This is the outcome that supports the drift result, alongside the cos(SZA) check.
- **A significant ρ at some PICS:** that site's patch systematically migrated across the detector over the record. Because responsivity and stray light vary cross-track, a monotonic column trend could imprint an apparent per-site drift. Note the *sign* and magnitude; a large |ρ| on a small n (watch the printed `n`) is fragile — confirm before drawing conclusions.
- **§5 tight (ρ near +1):** VZA faithfully proxies |cross-track offset|, so the retained VZA-vs-time check (available without cubes) is a sound unsigned substitute. The one thing it cannot see — a scene swinging symmetrically across nadir with little VZA change — is exactly what mean `col` here would catch.

> This is a diagnostic on cached pixel cubes, not a calibrated product. If cubes are missing for some scenes, the printed roster in §2 tells you which to re-extract (`scripts/run_single_patch_cube_extract.py` / the batched cube-extract) before re-running.